# Multimodal Integration: High-Content Imaging + Transcriptomics

**Goal**: Integrate Cell Painting imaging features with L1000 transcriptomic profiles using:
- Correlation analysis
- Partial Least Squares (PLS)
- Multi-Omics Factor Analysis (MOFA2)

**Data**: JUMP-CP (Cell Painting) + L1000 (transcriptomics) public datasets

---
## 1. Setup & Imports

In [ ]:
# Standard libraries
import os
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from scipy.cluster.hierarchy import linkage
from scipy.stats import spearmanr, pearsonr

# MOFA2
from mofapy2.run.entry_point import entry_point

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Create output directories
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'data').mkdir(exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)

print("✓ Setup complete")
print(f"✓ Output directory: {OUTPUT_DIR.absolute()}")

---
## 2. Data Download & Loading

We'll download JUMP-CP Cell Painting features and L1000 transcriptomics data from public sources.

In [ ]:
def load_data_files():
    """
    Load JUMP-CP + L1000 data from files.
    
    First run: python download_data.py to create the data files.
    
    Returns:
        tuple: (classical_features_df, embeddings_df, transcriptomics_df, metadata_df)
    """
    data_dir = Path('data/processed')
    
    print("Loading data files...")
    
    # Check if data exists
    required_files = [
        'metadata.csv',
        'classical_features.parquet',
        'embeddings.parquet',
        'transcriptomics.parquet'
    ]
    
    missing_files = [f for f in required_files if not (data_dir / f).exists()]
    
    if missing_files:
        print(f"\n⚠ Missing data files: {missing_files}")
        print("\nPlease run the data download script first:")
        print("  python download_data.py")
        print("\nFalling back to synthetic data generation...")
        return generate_synthetic_data()
    
    # Load all files
    metadata = pd.read_csv(data_dir / 'metadata.csv')
    classical_features = pd.read_parquet(data_dir / 'classical_features.parquet')
    embeddings = pd.read_parquet(data_dir / 'embeddings.parquet')
    transcriptomics = pd.read_parquet(data_dir / 'transcriptomics.parquet')
    
    print(f"  ✓ Loaded {len(metadata)} samples")
    print(f"  ✓ Classical features: {classical_features.shape[1]-1} dimensions")
    print(f"  ✓ Embeddings: {embeddings.shape[1]-1} dimensions")
    print(f"  ✓ Genes: {transcriptomics.shape[1]-1} dimensions")
    
    return classical_features, embeddings, transcriptomics, metadata


def generate_synthetic_data():
    """
    Generate synthetic data for testing (fallback if real data not available).
    
    Returns:
        tuple: (classical_features_df, embeddings_df, transcriptomics_df, metadata_df)
    """
    print("\nGenerating synthetic data...")
    
    np.random.seed(42)
    n_samples = 500
    n_classical_features = 1783
    n_embedding_dims = 512
    n_genes = 978
    
    # Generate sample metadata
    compounds = [f'COMPOUND_{i:03d}' for i in range(50)]
    plates = [f'PLATE_{i:02d}' for i in range(10)]
    wells = [f'{row}{col:02d}' for row in 'ABCDEFGH' for col in range(1, 13)]
    
    metadata = pd.DataFrame({
        'perturbation': np.random.choice(compounds, n_samples),
        'plate': np.random.choice(plates, n_samples),
        'well': np.random.choice(wells, n_samples),
        'replicate': np.random.randint(1, 4, n_samples),
        'dose_um': np.random.choice([0.1, 1.0, 10.0], n_samples),
        'cell_line': 'A549'
    })
    
    metadata['sample_key'] = (
        metadata['perturbation'] + '_' + 
        metadata['plate'] + '_' + 
        metadata['well'] + '_' + 
        metadata['replicate'].astype(str)
    )
    
    # Generate features
    classical_features = pd.DataFrame(
        np.random.randn(n_samples, n_classical_features),
        columns=[f'Cells_AreaShape_Feature_{i:03d}' if i < 15
                 else f'Cells_Intensity_Feature_{i:03d}' if i < 55
                 else f'Cells_Texture_Feature_{i:03d}' if i < 85
                 else f'Cells_Feature_{i:04d}'
                 for i in range(n_classical_features)]
    )
    classical_features['sample_key'] = metadata['sample_key']
    
    embeddings = pd.DataFrame(
        np.random.randn(n_samples, n_embedding_dims),
        columns=[f'Embedding_{i:03d}' for i in range(n_embedding_dims)]
    )
    embeddings['sample_key'] = metadata['sample_key']
    
    # Use some real L1000 landmark genes
    real_genes = [
        'TP53', 'MYC', 'EGFR', 'VEGFA', 'TNF', 'IL6', 'CDKN1A', 'BCL2',
        'GAPDH', 'ACTB', 'JUN', 'FOS', 'ATF3', 'DUSP1', 'FOSB', 'EGR1'
    ]
    gene_cols = real_genes + [f'Gene_{i:03d}' for i in range(len(real_genes), n_genes)]
    
    transcriptomics = pd.DataFrame(
        np.random.randn(n_samples, n_genes) * 2 + np.random.randn(1, n_genes),
        columns=gene_cols
    )
    transcriptomics['sample_key'] = metadata['sample_key']
    
    print(f"  ✓ Generated {n_samples} samples")
    print(f"  ✓ Classical features: {n_classical_features} dimensions")
    print(f"  ✓ Embeddings: {n_embedding_dims} dimensions")
    print(f"  ✓ Genes: {n_genes} dimensions")
    
    return classical_features, embeddings, transcriptomics, metadata


# Load data (will fallback to synthetic if files don't exist)
classical_features_raw, embeddings_raw, transcriptomics_raw, metadata = load_data_files()

print("\n✓ Data loading complete")

---
## 3. Metadata Alignment

Align all modalities using sample keys (perturbation + plate + well + replicate).

In [ ]:
def build_sample_key(df, key_columns=['perturbation', 'plate', 'well', 'replicate']):
    """
    Build a unique sample key from metadata columns.
    
    Args:
        df: DataFrame with metadata columns
        key_columns: List of columns to combine
        
    Returns:
        Series: Unique sample keys
    """
    return df[key_columns].astype(str).agg('_'.join, axis=1)


def align_modalities(metadata, classical, embeddings, transcriptomics):
    """
    Align all modalities to a common set of samples.
    
    Args:
        metadata: Sample metadata DataFrame
        classical: Classical features DataFrame
        embeddings: Embedding features DataFrame
        transcriptomics: Gene expression DataFrame
        
    Returns:
        tuple: (aligned_metadata, aligned_classical, aligned_embeddings, aligned_transcriptomics)
    """
    print("Aligning modalities...")
    
    # Find common samples across all modalities
    common_samples = set(metadata['sample_key'])
    common_samples &= set(classical['sample_key'])
    common_samples &= set(embeddings['sample_key'])
    common_samples &= set(transcriptomics['sample_key'])
    
    common_samples = sorted(common_samples)
    print(f"  ✓ Found {len(common_samples)} samples with all modalities")
    
    # Filter to common samples
    metadata_aligned = metadata[metadata['sample_key'].isin(common_samples)].copy()
    classical_aligned = classical[classical['sample_key'].isin(common_samples)].copy()
    embeddings_aligned = embeddings[embeddings['sample_key'].isin(common_samples)].copy()
    transcriptomics_aligned = transcriptomics[transcriptomics['sample_key'].isin(common_samples)].copy()
    
    # Sort all by sample_key to ensure alignment
    metadata_aligned = metadata_aligned.sort_values('sample_key').reset_index(drop=True)
    classical_aligned = classical_aligned.sort_values('sample_key').reset_index(drop=True)
    embeddings_aligned = embeddings_aligned.sort_values('sample_key').reset_index(drop=True)
    transcriptomics_aligned = transcriptomics_aligned.sort_values('sample_key').reset_index(drop=True)
    
    # Verify alignment
    assert all(metadata_aligned['sample_key'] == classical_aligned['sample_key'])
    assert all(metadata_aligned['sample_key'] == embeddings_aligned['sample_key'])
    assert all(metadata_aligned['sample_key'] == transcriptomics_aligned['sample_key'])
    
    print("  ✓ All modalities aligned successfully")
    
    return metadata_aligned, classical_aligned, embeddings_aligned, transcriptomics_aligned


# Align all modalities
metadata, classical_features, embeddings, transcriptomics = align_modalities(
    metadata, classical_features_raw, embeddings_raw, transcriptomics_raw
)

print(f"\n✓ Final dataset shape: {len(metadata)} samples")
print(f"  - Classical features: {classical_features.shape[1]-1} dimensions")
print(f"  - Embeddings: {embeddings.shape[1]-1} dimensions")
print(f"  - Genes: {transcriptomics.shape[1]-1} dimensions")

---
## 4. Preprocessing & Cleaning

In [ ]:
def remove_na_features(df, threshold=0.05):
    """
    Remove features with too many NA values.
    
    Args:
        df: DataFrame with features
        threshold: Maximum fraction of NA values allowed
        
    Returns:
        DataFrame: Cleaned DataFrame
    """
    na_fraction = df.isna().mean()
    keep_cols = na_fraction[na_fraction <= threshold].index
    removed = len(df.columns) - len(keep_cols)
    if removed > 0:
        print(f"  - Removed {removed} features with >{threshold*100}% NA values")
    return df[keep_cols]


def zscore_matrix(df, feature_cols):
    """
    Z-score normalize features across samples.
    
    Args:
        df: DataFrame with features
        feature_cols: List of feature column names
        
    Returns:
        DataFrame: Normalized DataFrame
    """
    df_normalized = df.copy()
    scaler = StandardScaler()
    df_normalized[feature_cols] = scaler.fit_transform(df[feature_cols])
    return df_normalized


def remove_low_variance_features(df, feature_cols, threshold=0.01):
    """
    Remove features with very low variance.
    
    Args:
        df: DataFrame with features
        feature_cols: List of feature column names
        threshold: Minimum variance threshold
        
    Returns:
        DataFrame: Filtered DataFrame
    """
    variances = df[feature_cols].var()
    keep_cols = variances[variances > threshold].index.tolist()
    removed = len(feature_cols) - len(keep_cols)
    if removed > 0:
        print(f"  - Removed {removed} low-variance features")
    
    # Keep non-feature columns
    non_feature_cols = [col for col in df.columns if col not in feature_cols]
    return df[non_feature_cols + keep_cols]


print("Preprocessing classical features...")
feature_cols_classical = [col for col in classical_features_raw.columns if col != 'sample_key']
classical_features = remove_na_features(classical_features_raw[['sample_key'] + feature_cols_classical])
feature_cols_classical = [col for col in classical_features.columns if col != 'sample_key']
classical_features = remove_low_variance_features(classical_features, feature_cols_classical)
feature_cols_classical = [col for col in classical_features.columns if col != 'sample_key']
classical_features = zscore_matrix(classical_features, feature_cols_classical)
print(f"  ✓ Final classical features: {len(feature_cols_classical)} dimensions")

print("\nPreprocessing embeddings...")
feature_cols_embeddings = [col for col in embeddings_raw.columns if col != 'sample_key']
embeddings = remove_na_features(embeddings_raw[['sample_key'] + feature_cols_embeddings])
feature_cols_embeddings = [col for col in embeddings.columns if col != 'sample_key']
embeddings = remove_low_variance_features(embeddings, feature_cols_embeddings)
feature_cols_embeddings = [col for col in embeddings.columns if col != 'sample_key']
embeddings = zscore_matrix(embeddings, feature_cols_embeddings)
print(f"  ✓ Final embeddings: {len(feature_cols_embeddings)} dimensions")

print("\nPreprocessing transcriptomics...")
feature_cols_genes = [col for col in transcriptomics_raw.columns if col != 'sample_key']
transcriptomics = remove_na_features(transcriptomics_raw[['sample_key'] + feature_cols_genes])
feature_cols_genes = [col for col in transcriptomics.columns if col != 'sample_key']
transcriptomics = remove_low_variance_features(transcriptomics, feature_cols_genes)
feature_cols_genes = [col for col in transcriptomics.columns if col != 'sample_key']
transcriptomics = zscore_matrix(transcriptomics, feature_cols_genes)
print(f"  ✓ Final genes: {len(feature_cols_genes)} dimensions")

print("\n✓ Preprocessing complete")

In [ ]:
# Construct clean matrices for analysis
X_hci_classical = classical_features[feature_cols_classical].values
X_hci_embeddings = embeddings[feature_cols_embeddings].values
Y_transcriptomics = transcriptomics[feature_cols_genes].values

print(f"Analysis matrices:")
print(f"  X_hci_classical: {X_hci_classical.shape}")
print(f"  X_hci_embeddings: {X_hci_embeddings.shape}")
print(f"  Y_transcriptomics: {Y_transcriptomics.shape}")

---
## 5. Correlation Analysis

**Goal**: Make embeddings interpretable by correlating with classical features.

In [ ]:
def compute_correlation_matrix(A, B, method='pearson'):
    """
    Compute correlation matrix between two feature sets.
    
    Args:
        A: Array (n_samples, n_features_A)
        B: Array (n_samples, n_features_B)
        method: 'pearson' or 'spearman'
        
    Returns:
        Array: Correlation matrix (n_features_A, n_features_B)
    """
    n_features_A = A.shape[1]
    n_features_B = B.shape[1]
    corr_matrix = np.zeros((n_features_A, n_features_B))
    
    for i in range(n_features_A):
        for j in range(n_features_B):
            if method == 'pearson':
                corr_matrix[i, j] = pearsonr(A[:, i], B[:, j])[0]
            else:
                corr_matrix[i, j] = spearmanr(A[:, i], B[:, j])[0]
    
    return corr_matrix


print("Computing correlation matrix: embeddings × classical features...")
# Subsample for speed (correlate embeddings with classical features)
n_embedding_subsample = min(100, len(feature_cols_embeddings))
n_classical_subsample = min(200, len(feature_cols_classical))

embedding_indices = np.linspace(0, len(feature_cols_embeddings)-1, n_embedding_subsample, dtype=int)
classical_indices = np.linspace(0, len(feature_cols_classical)-1, n_classical_subsample, dtype=int)

corr_matrix = compute_correlation_matrix(
    X_hci_embeddings[:, embedding_indices],
    X_hci_classical[:, classical_indices],
    method='pearson'
)

print(f"  ✓ Correlation matrix shape: {corr_matrix.shape}")
print(f"  ✓ Mean absolute correlation: {np.abs(corr_matrix).mean():.3f}")

In [ ]:
def plot_correlation_clustermap(corr_matrix, embedding_labels, classical_labels, output_path):
    """
    Create hierarchical clustering heatmap of correlations.
    
    Args:
        corr_matrix: Correlation matrix
        embedding_labels: Row labels
        classical_labels: Column labels
        output_path: Path to save figure
    """
    fig = plt.figure(figsize=(14, 10))
    
    # Create clustermap
    g = sns.clustermap(
        corr_matrix,
        cmap='RdBu_r',
        center=0,
        vmin=-0.5,
        vmax=0.5,
        xticklabels=False,
        yticklabels=False,
        figsize=(14, 10),
        cbar_kws={'label': 'Pearson Correlation'}
    )
    
    g.ax_heatmap.set_xlabel('Classical Cell Painting Features', fontsize=12)
    g.ax_heatmap.set_ylabel('Embedding Dimensions', fontsize=12)
    plt.suptitle('Embedding–Classical Feature Correlation', y=0.98, fontsize=14, fontweight='bold')
    
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.close()


# Plot correlation clustermap
embedding_labels = [feature_cols_embeddings[i] for i in embedding_indices]
classical_labels = [feature_cols_classical[i] for i in classical_indices]

plot_correlation_clustermap(
    corr_matrix,
    embedding_labels,
    classical_labels,
    OUTPUT_DIR / 'figures' / 'correlation_clustermap.png'
)

In [ ]:
def summarize_embedding_interpretation(corr_matrix, embedding_labels, classical_labels, top_k=5):
    """
    Identify top-correlated classical features for each embedding dimension.
    
    Args:
        corr_matrix: Correlation matrix (embeddings × classical)
        embedding_labels: Embedding dimension names
        classical_labels: Classical feature names
        top_k: Number of top features to report
        
    Returns:
        DataFrame: Summary table
    """
    summaries = []
    
    for i, emb_label in enumerate(embedding_labels):
        # Get top absolute correlations
        abs_corrs = np.abs(corr_matrix[i, :])
        top_indices = np.argsort(abs_corrs)[-top_k:][::-1]
        
        top_features = [classical_labels[idx] for idx in top_indices]
        top_corrs = [corr_matrix[i, idx] for idx in top_indices]
        
        summaries.append({
            'embedding_dim': emb_label,
            'mean_abs_correlation': abs_corrs.mean(),
            'max_abs_correlation': abs_corrs.max(),
            'top_correlated_features': ', '.join([f"{feat} ({corr:.2f})" 
                                                   for feat, corr in zip(top_features, top_corrs)])
        })
    
    return pd.DataFrame(summaries)


# Generate interpretation summary
embedding_interpretation = summarize_embedding_interpretation(
    corr_matrix,
    embedding_labels,
    classical_labels,
    top_k=3
)

print("\nTop interpretable embedding dimensions:")
print(embedding_interpretation.nlargest(10, 'mean_abs_correlation')[[
    'embedding_dim', 'mean_abs_correlation', 'max_abs_correlation'
]])

embedding_interpretation.to_csv(
    OUTPUT_DIR / 'data' / 'embedding_interpretation.csv',
    index=False
)
print(f"\n✓ Saved: {OUTPUT_DIR / 'data' / 'embedding_interpretation.csv'}")

---
## 6. PLS Integration

**Goal**: Find latent components that maximize covariance between imaging and transcriptomics.

### Understanding PLS Dimensions

PLS (Partial Least Squares) regression finds latent components that maximize covariance between two data matrices.

**Data Flow and Dimensions:**

```
Input Data:
  X (imaging):           n_samples × n_features_X    (e.g., 180 × 642)
  Y (transcriptomics):   n_samples × n_features_Y    (e.g., 180 × 978 → 180 after PCA)

PCA Reduction (optional):
  Y_original:            180 × 978 genes
  Y_reduced:             180 × 180 PCs  (max components = min(n_samples, n_genes))
  
PLS Model Fit:
  pls.fit(X, Y_reduced)
  
PLS Components:        min(n_samples, n_features_X, n_features_Y)
  Practical limit:      min(180, 642, 180) = 180
  Chosen:               5 components (for interpretability)

PLS Outputs:
  X_scores:             180 × 5      (latent imaging scores)
  Y_predicted:          180 × 180    (predicted transcriptomics)
  X_loadings:           642 × 5      (imaging feature weights)
  Y_loadings:           180 × 5      (transcriptomics feature weights)

Key Operations:
  pls.transform(X)  →  X_scores (180 × 5)     ✓ Transform imaging to latent space
  pls.predict(X)    →  Y_pred (180 × 180)     ✓ Predict transcriptomics from imaging
  pls.transform(Y)  →  ERROR!                  ✗ Y cannot be transformed (only predicted)
```

**Important Constraints:**

1. **PCA components:** `n_components ≤ min(n_samples, n_features)`
   - With 180 samples and 978 genes: max 180 components
   
2. **PLS components:** `n_components ≤ min(n_samples, n_features_X, n_features_Y)`
   - With 180 samples, 642 imaging features, 180 transcriptomics features: max 180 components
   - We use 5 for interpretability and to avoid overfitting
   
3. **Transform vs Predict:**
   - `pls.transform(X)`: Projects X onto latent components (returns n_samples × n_components)
   - `pls.predict(X)`: Predicts Y from X (returns n_samples × n_features_Y)
   - Only X can be transformed; Y can only be predicted

**Why PCA Before PLS?**

- Original transcriptomics: 978 genes
- This creates a very wide Y matrix relative to sample size
- PCA reduces dimensionality while preserving variance
- Makes PLS more stable and computationally efficient
- Reduces risk of overfitting

In [ ]:
def run_pls(X, Y, n_components=5):
    """
    Run PLS regression to find latent components.
    
    Args:
        X: Predictor matrix (n_samples, n_features_X)
        Y: Response matrix (n_samples, n_features_Y)
        n_components: Number of PLS components
        
    Returns:
        dict: PLS results including scores, loadings, variance explained
    """
    print(f"Running PLS with {n_components} components...")
    
    pls = PLSRegression(n_components=n_components, scale=False)  # Already z-scored
    pls.fit(X, Y)
    
    # Get latent scores
    X_scores = pls.transform(X)  # U: imaging latent scores (n_samples, n_components)
    
    # Calculate variance explained by X
    X_var_explained = np.var(X_scores, axis=0) / np.var(X, axis=0).sum()
    
    # Calculate cumulative R² for Y (how well we predict Y)
    Y_pred = pls.predict(X)
    r2_total = 1 - np.sum((Y - Y_pred)**2) / np.sum((Y - Y.mean(axis=0))**2)
    
    # Per-component R² (approximate by fitting models with increasing components)
    Y_var_explained = []
    for i in range(1, n_components + 1):
        pls_temp = PLSRegression(n_components=i, scale=False)
        pls_temp.fit(X, Y)
        Y_pred_temp = pls_temp.predict(X)
        r2 = 1 - np.sum((Y - Y_pred_temp)**2) / np.sum((Y - Y.mean(axis=0))**2)
        Y_var_explained.append(r2)
    
    results = {
        'model': pls,
        'X_scores': X_scores,
        'Y_predicted': Y_pred,
        'X_loadings': pls.x_loadings_,
        'Y_loadings': pls.y_loadings_,
        'X_var_explained': X_var_explained,
        'Y_var_explained': np.array(Y_var_explained),
        'Y_r2_total': r2_total,
        'coef': pls.coef_
    }
    
    print(f"  ✓ X variance explained: {X_var_explained[:min(3, len(X_var_explained))]}")
    print(f"  ✓ Y variance explained (cumulative R²): {Y_var_explained[:min(3, len(Y_var_explained))]}")
    print(f"  ✓ Total Y R²: {r2_total:.4f}")
    
    return results


# Concatenate imaging features
X_imaging_combined = np.hstack([X_hci_classical, X_hci_embeddings])
print(f"Combined imaging features: {X_imaging_combined.shape}")

# Optional: PCA reduction of transcriptomics for speed
USE_PCA_REDUCTION = True
DESIRED_PCA_COMPONENTS = 200

if USE_PCA_REDUCTION and Y_transcriptomics.shape[1] > DESIRED_PCA_COMPONENTS:
    # Calculate maximum possible components
    n_samples, n_features = Y_transcriptomics.shape
    max_components = min(n_samples, n_features)
    n_pca_components = min(DESIRED_PCA_COMPONENTS, max_components)
    
    print(f"\nReducing transcriptomics with PCA ({n_pca_components} components)...")
    print(f"  (max possible: {max_components}, desired: {DESIRED_PCA_COMPONENTS})")
    
    pca = PCA(n_components=n_pca_components, random_state=42)
    Y_transcriptomics_reduced = pca.fit_transform(Y_transcriptomics)
    print(f"  ✓ Reduced from {n_features} to {n_pca_components} features")
    print(f"  ✓ Explained variance: {pca.explained_variance_ratio_.sum():.2%}")
else:
    Y_transcriptomics_reduced = Y_transcriptomics
    print(f"\nSkipping PCA reduction (features: {Y_transcriptomics.shape[1]})")

# Run PLS
pls_results = run_pls(X_imaging_combined, Y_transcriptomics_reduced, n_components=5)

In [ ]:
def plot_pls_variance_explained(pls_results, output_path):
    """
    Plot variance explained by PLS components.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # X variance
    axes[0].bar(range(1, len(pls_results['X_var_explained'])+1),
                pls_results['X_var_explained'],
                color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Component', fontsize=12)
    axes[0].set_ylabel('Variance Explained', fontsize=12)
    axes[0].set_title('Imaging Features (X)', fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Y variance (cumulative R²)
    axes[1].plot(range(1, len(pls_results['Y_var_explained'])+1),
                 pls_results['Y_var_explained'],
                 marker='o', linewidth=2, markersize=8, color='coral')
    axes[1].set_xlabel('Component', fontsize=12)
    axes[1].set_ylabel('Cumulative R²', fontsize=12)
    axes[1].set_title('Transcriptomics (Y)', fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.close()


plot_pls_variance_explained(
    pls_results,
    OUTPUT_DIR / 'figures' / 'pls_variance_explained.png'
)

In [ ]:
def plot_pls_biplots(pls_results, metadata, output_path):
    """
    Create biplots for top 2 PLS components.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    X_scores = pls_results['X_scores']
    
    # Color by perturbation (sample first 10 for legend clarity)
    unique_perts = metadata['perturbation'].unique()[:10]
    colors = metadata['perturbation'].map({
        pert: f'C{i}' for i, pert in enumerate(unique_perts)
    }).fillna('lightgray')
    
    # Component 1 vs 2
    axes[0].scatter(X_scores[:, 0], X_scores[:, 1], 
                   c=colors, alpha=0.6, s=30, edgecolors='k', linewidth=0.3)
    axes[0].set_xlabel('Component 1', fontsize=12)
    axes[0].set_ylabel('Component 2', fontsize=12)
    axes[0].set_title('PLS Score Plot (Components 1-2)', fontweight='bold')
    axes[0].grid(alpha=0.3)
    axes[0].axhline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
    axes[0].axvline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Component 2 vs 3
    axes[1].scatter(X_scores[:, 1], X_scores[:, 2],
                   c=colors, alpha=0.6, s=30, edgecolors='k', linewidth=0.3)
    axes[1].set_xlabel('Component 2', fontsize=12)
    axes[1].set_ylabel('Component 3', fontsize=12)
    axes[1].set_title('PLS Score Plot (Components 2-3)', fontweight='bold')
    axes[1].grid(alpha=0.3)
    axes[1].axhline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
    axes[1].axvline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.close()


plot_pls_biplots(
    pls_results,
    metadata,
    OUTPUT_DIR / 'figures' / 'pls_biplots.png'
)

In [ ]:
# Save PLS results
pls_scores_df = pd.DataFrame(
    pls_results['X_scores'],
    columns=[f'PLS_Component_{i+1}' for i in range(pls_results['X_scores'].shape[1])]
)
pls_scores_df['sample_key'] = metadata['sample_key'].values
pls_scores_df.to_csv(OUTPUT_DIR / 'data' / 'pls_scores.csv', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'data' / 'pls_scores.csv'}")

# Save top loadings
all_feature_names = feature_cols_classical + feature_cols_embeddings
pls_loadings_df = pd.DataFrame(
    pls_results['X_loadings'],
    columns=[f'PLS_Component_{i+1}' for i in range(pls_results['X_loadings'].shape[1])]
)
pls_loadings_df['feature'] = all_feature_names
pls_loadings_df.to_csv(OUTPUT_DIR / 'data' / 'pls_loadings.csv', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'data' / 'pls_loadings.csv'}")

---
## 7. MOFA2 Integration

**Goal**: Multi-omics factor analysis to discover shared and unique variation.

### Understanding MOFA2

**MOFA2** (Multi-Omics Factor Analysis v2) is a probabilistic framework for discovering interpretable low-dimensional representations from multi-view datasets.

### Theory & Model

**Core Concept:**
MOFA learns a set of **latent factors** (Z) that explain variation across multiple data modalities (views). Unlike PLS which focuses on maximizing covariance, MOFA decomposes variance into:
- **Shared variation**: captured by factors active in multiple views
- **Unique variation**: captured by factors active in single views

**Mathematical Model:**

For each view `m` and sample `n`:

```
Y_m,n = W_m × Z_n + ε_m,n

where:
  Y_m,n  = observed data (features × samples) for view m
  W_m    = weight matrix (features × factors) for view m
  Z_n    = factor values (factors × samples) - shared across views
  ε_m,n  = noise/residual
```

**Key Features:**

1. **Bayesian Framework**: Uses variational inference with priors that encourage sparsity
2. **Automatic Relevance Determination (ARD)**: Automatically determines which factors are active
3. **Spike-and-Slab Priors**: Identify which features contribute to each factor
4. **Group Structure**: Can handle multiple sample groups (e.g., conditions, batches)

### MOFA2 vs PLS

| Aspect | PLS | MOFA2 |
|--------|-----|-------|
| Objective | Maximize covariance between X and Y | Decompose variance across views |
| Supervision | Supervised (X predicts Y) | Unsupervised |
| Sparsity | No built-in sparsity | Sparse weights via priors |
| Factor Selection | User-defined | Automatic (ARD) |
| Interpretability | Moderate | High (sparse loadings) |
| Computation | Fast | Slower (iterative) |

### Data Structure Requirements (CRITICAL!)

**MOFA2 expects data in this exact format:**

```python
# Structure: data[view][group]
# Each element is a numpy array with shape (n_samples, n_features)

# For M views and G groups:
data = [
    [view_0_group_0, view_0_group_1, ..., view_0_group_G-1],  # View 0
    [view_1_group_0, view_1_group_1, ..., view_1_group_G-1],  # View 1
    ...
    [view_M-1_group_0, view_M-1_group_1, ..., view_M-1_group_G-1]  # View M-1
]

# For our case (2 views, 1 group):
data = [
    [imaging_array],         # View 0 (Imaging), Group 0
    [transcriptomics_array]  # View 1 (Transcriptomics), Group 0
]
```

**CRITICAL Requirements:**
- **Type**: Numpy arrays (NOT DataFrames!)
- **Shape**: `(n_samples, n_features)` - rows are samples, columns are features
- **Orientation**: Do NOT transpose! Keep samples as rows
- **Nesting**: Must be `data[view][group]` structure

### Common Mistakes to Avoid

```python
# ❌ WRONG: Using DataFrames
data = [[imaging_df, transcriptomics_df]]

# ❌ WRONG: Transposed (features × samples)
data = [[imaging.T, transcriptomics.T]]

# ❌ WRONG: Wrong nesting (group then view)
data = [[imaging, transcriptomics]]  # This is 1 view, 2 groups!

# ✓ CORRECT: Numpy arrays, proper orientation, correct nesting
data = [
    [imaging_array],           # Shape: (180, 642)
    [transcriptomics_array]    # Shape: (180, 180)
]
```

### Dimension Flow

```
Input Arrays (NOT transposed):
  View 0 (Imaging):        180 samples × 642 features
  View 1 (Transcriptomics): 180 samples × 180 features

MOFA2 Structure:
  data[0][0] = imaging_array        (180, 642)
  data[1][0] = transcriptomics_array (180, 180)

MOFA2 Model:
  Factors (Z):             180 samples × 10 factors
  Weights_imaging (W0):    642 features × 10 factors
  Weights_transcriptomics (W1): 180 features × 10 factors

Reconstruction:
  View 0 = Z × W0^T  →  180 × 642
  View 1 = Z × W1^T  →  180 × 180
```

### Model Parameters

**Likelihoods:**
- `'gaussian'`: For continuous data (our case)
- `'poisson'`: For count data (e.g., scRNA-seq)
- `'bernoulli'`: For binary data

**Priors:**
- `spikeslab_weights=True`: Encourages sparse feature weights
- `ard_weights=True`: Automatic relevance determination for factors

**Training Options:**
- `iter`: Maximum iterations (we use 1000)
- `convergence_mode='fast'`: Balance between speed and accuracy
- `dropR2=0.001`: Minimum variance explained to keep a factor
- `seed=42`: Reproducibility

### Outputs & Interpretation

**Factor Matrix (Z):**
- Dimension: n_samples × n_factors
- Interpretation: Sample coordinates in latent space
- Can be correlated with metadata (treatment, dose, time)

**Weight Matrices (W):**
- Dimension: n_features × n_factors per view
- Interpretation: Feature importance for each factor
- High absolute weights → feature drives that factor

**Variance Explained (R²):**
- Per factor, per view
- Identifies which factors are:
  - **Shared**: High R² in multiple views
  - **Specific**: High R² in one view only

### Use Cases

**MOFA2 is ideal when you want to:**
1. Identify shared vs. view-specific variation
2. Discover sparse, interpretable factors
3. Handle missing data across views
4. Compare variation across multiple conditions/groups
5. Prioritize features driving biological processes

**Example Interpretation:**
```
Factor 1: High R² in both imaging & transcriptomics
  → Shared biological process (e.g., cell cycle)
  → Top imaging features: cell size, nuclear area
  → Top genes: CCNB1, CDK1, MKI67

Factor 2: High R² in imaging only
  → Morphological variation independent of transcription
  → Top features: cell shape, texture
  
Factor 3: High R² in transcriptomics only
  → Transcriptional program not reflected in morphology
  → Top genes: stress response pathway
```

In [ ]:
def run_mofa2(view1, view2, view1_name='Imaging', view2_name='Transcriptomics', 
              n_factors=10, output_dir=None):
    """
    Run MOFA2 on two data views.
    
    Args:
        view1: Array (n_samples, n_features_1)
        view2: Array (n_samples, n_features_2)
        view1_name: Name of first view
        view2_name: Name of second view
        n_factors: Number of factors to learn
        output_dir: Directory to save model
        
    Returns:
        dict: MOFA2 results
    """
    print(f"Running MOFA2 with {n_factors} factors...")
    
    # Initialize MOFA entry point
    ent = entry_point()
    
    # MOFA2 data structure: data[view][group]
    # For M=2 views, G=1 group: [[view1_g1], [view2_g1]]
    # Each element is numpy array with shape (n_samples, n_features)
    # IMPORTANT: rows=samples, columns=features (NOT transposed!)
    
    data = [
        [view1],  # View 1, Group 1
        [view2]   # View 2, Group 1
    ]
    
    # Set data matrix with likelihoods
    ent.set_data_matrix(data, likelihoods=['gaussian', 'gaussian'])
    
    # Set custom view names (override auto-generated names)
    ent.data_opts['views_names'] = [view1_name, view2_name]
    
    # Set model options
    ent.set_model_options(
        factors=n_factors,
        spikeslab_weights=True,
        ard_weights=True
    )
    
    # Set training options
    ent.set_train_options(
        iter=1000,
        convergence_mode='fast',
        dropR2=0.001,
        verbose=False,
        seed=42
    )
    
    # Build and train model
    print("  Building MOFA2 model...")
    ent.build()
    print("  Training MOFA2 model (this may take a minute)...")
    ent.run()
    
    # Save model if output_dir provided
    if output_dir:
        model_path = output_dir / 'models' / 'mofa_model.hdf5'
        ent.save(str(model_path))
        print(f"  ✓ Model saved: {model_path}")
    
    # Extract results
    # Access weights using view indices (0, 1) not names
    factors = ent.model.nodes['Z'].getExpectation()
    
    # Get weights for each view - MOFA2 uses indices internally
    W_node = ent.model.nodes['W']
    weights_list = W_node.getExpectations()  # Returns list of weight matrices
    
    weights_view1 = weights_list[0]  # View 0
    weights_view2 = weights_list[1]  # View 1
    
    # Calculate variance explained
    r2_total = ent.model.calculate_variance_explained()
    
    results = {
        'model': ent,
        'factors': factors,
        'weights': {
            view1_name: weights_view1,
            view2_name: weights_view2
        },
        'r2': r2_total,
        'view_names': [view1_name, view2_name]
    }
    
    print(f"  ✓ MOFA2 training complete")
    print(f"  ✓ Active factors: {factors.shape[1]}")
    
    return results


# Run MOFA2
mofa_results = run_mofa2(
    X_imaging_combined,
    Y_transcriptomics_reduced,
    view1_name='Imaging',
    view2_name='Transcriptomics',
    n_factors=10,
    output_dir=OUTPUT_DIR
)

In [ ]:
def plot_mofa_variance_explained(mofa_results, output_path):
    """
    Plot variance explained by MOFA factors per view.
    """
    r2 = mofa_results['r2']
    view_names = mofa_results['view_names']
    
    # MOFA2 R2 structure: r2[view] is a dictionary with factor keys
    # Need to handle the actual key format from MOFA2
    
    # Get first view to determine number of factors
    first_view = view_names[0]
    factor_keys = sorted([k for k in r2[first_view].keys() if 'Factor' in str(k)])
    n_factors = len(factor_keys)
    
    # Extract R2 values into matrix
    r2_matrix = []
    for view in view_names:
        view_r2 = []
        for factor_key in factor_keys:
            # MOFA2 might use 'Factor1', 'Factor2', etc.
            r2_val = r2[view].get(factor_key, 0.0)
            view_r2.append(r2_val)
        r2_matrix.append(view_r2)
    
    r2_matrix = np.array(r2_matrix)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Heatmap
    sns.heatmap(
        r2_matrix,
        annot=True,
        fmt='.1f',
        cmap='YlOrRd',
        yticklabels=view_names,
        xticklabels=[f'F{i+1}' for i in range(n_factors)],
        cbar_kws={'label': 'Variance Explained (%)'},
        ax=axes[0]
    )
    axes[0].set_title('MOFA2 Variance Explained per Factor', fontweight='bold')
    axes[0].set_xlabel('Factor', fontsize=12)
    axes[0].set_ylabel('View', fontsize=12)
    
    # Bar plot - total per factor
    total_r2 = r2_matrix.sum(axis=0)
    axes[1].bar(range(1, n_factors+1), total_r2, color='steelblue', alpha=0.7)
    axes[1].set_xlabel('Factor', fontsize=12)
    axes[1].set_ylabel('Total Variance Explained (%)', fontsize=12)
    axes[1].set_title('Total Variance per Factor', fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.close()


plot_mofa_variance_explained(
    mofa_results,
    OUTPUT_DIR / 'figures' / 'mofa_variance_explained.png'
)

In [ ]:
def plot_mofa_factor_heatmap(mofa_results, metadata, output_path):
    """
    Plot heatmap of MOFA factors across samples.
    """
    factors = mofa_results['factors']
    
    # Create clustermap (creates its own figure)
    g = sns.clustermap(
        factors.T,
        cmap='RdBu_r',
        center=0,
        xticklabels=False,
        yticklabels=[f'Factor {i+1}' for i in range(factors.shape[1])],
        figsize=(14, 8),
        cbar_kws={'label': 'Factor Value'},
        col_cluster=True,
        row_cluster=True
    )
    
    g.ax_heatmap.set_xlabel('Samples', fontsize=12)
    g.ax_heatmap.set_ylabel('Factor', fontsize=12)
    g.fig.suptitle('MOFA2 Factor Heatmap', y=0.98, fontsize=14, fontweight='bold')
    
    g.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.close(g.fig)


plot_mofa_factor_heatmap(
    mofa_results,
    metadata,
    OUTPUT_DIR / 'figures' / 'mofa_factor_heatmap.png'
)

In [ ]:
def analyze_factor_metadata_association(factors, metadata, categorical_cols=['perturbation']):
    """
    Analyze associations between MOFA factors and metadata.
    
    Args:
        factors: Factor matrix (n_samples, n_factors)
        metadata: Sample metadata DataFrame
        categorical_cols: Categorical columns to test
        
    Returns:
        DataFrame: Association statistics
    """
    from scipy.stats import f_oneway
    
    associations = []
    
    for factor_idx in range(factors.shape[1]):
        factor_values = factors[:, factor_idx]
        
        for col in categorical_cols:
            if col not in metadata.columns:
                continue
                
            # Group factor values by metadata category
            # Use positional indexing to avoid index alignment issues
            groups = []
            for category in metadata[col].unique():
                mask = metadata[col] == category
                category_factors = factor_values[mask]
                if len(category_factors) > 0:
                    groups.append(category_factors)
            
            # ANOVA test (need at least 2 groups)
            if len(groups) > 1:
                f_stat, p_value = f_oneway(*groups)
                
                associations.append({
                    'factor': f'Factor {factor_idx + 1}',
                    'metadata_column': col,
                    'f_statistic': f_stat,
                    'p_value': p_value,
                    'n_groups': len(groups)
                })
    
    return pd.DataFrame(associations)


# Analyze factor-metadata associations
factor_metadata = analyze_factor_metadata_association(
    mofa_results['factors'],
    metadata,
    categorical_cols=['perturbation']
)

print("\nTop factor-metadata associations:")
if len(factor_metadata) > 0:
    print(factor_metadata.nsmallest(10, 'p_value')[[
        'factor', 'metadata_column', 'f_statistic', 'p_value'
    ]])
else:
    print("No associations found")

In [ ]:
# Save MOFA results
mofa_factors_df = pd.DataFrame(
    mofa_results['factors'],
    columns=[f'MOFA_Factor_{i+1}' for i in range(mofa_results['factors'].shape[1])]
)
mofa_factors_df['sample_key'] = metadata['sample_key'].values
mofa_factors_df.to_csv(OUTPUT_DIR / 'data' / 'mofa_factors.csv', index=False)
print(f"✓ Saved: {OUTPUT_DIR / 'data' / 'mofa_factors.csv'}")

# Save weights
for view_name, weights in mofa_results['weights'].items():
    weights_df = pd.DataFrame(
        weights,
        columns=[f'MOFA_Factor_{i+1}' for i in range(weights.shape[1])]
    )
    output_file = OUTPUT_DIR / 'data' / f'mofa_weights_{view_name.lower()}.csv'
    weights_df.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")

---
## 8. Interpretation & Visualizations

**Summary**: Cross-method comparison and biological interpretation.

In [ ]:
def create_integration_summary(pls_results, mofa_results, embedding_interpretation, output_path):
    """
    Create comprehensive summary of multimodal integration.
    
    Args:
        pls_results: PLS analysis results
        mofa_results: MOFA2 results
        embedding_interpretation: Embedding correlation summary
        output_path: Path to save summary
    """
    summary = []
    
    # Top embedding dimensions by interpretability
    top_embeddings = embedding_interpretation.nlargest(
        10, 'mean_abs_correlation'
    )[['embedding_dim', 'mean_abs_correlation', 'top_correlated_features']]
    
    for _, row in top_embeddings.iterrows():
        summary.append({
            'embedding_dim': row['embedding_dim'],
            'interpretability_score': row['mean_abs_correlation'],
            'top_correlated_features': row['top_correlated_features'],
            'top_genes': 'N/A',  # Placeholder for gene enrichment
            'biological_annotation': 'Placeholder - perform pathway enrichment'
        })
    
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(output_path, index=False)
    print(f"✓ Saved: {output_path}")
    
    return summary_df


# Create summary
integration_summary = create_integration_summary(
    pls_results,
    mofa_results,
    embedding_interpretation,
    OUTPUT_DIR / 'data' / 'summary_embedding_biology.csv'
)

print("\nIntegration Summary (Top 5 Embeddings):")
print(integration_summary[['embedding_dim', 'interpretability_score']].head())

In [ ]:
def plot_method_comparison(pls_results, mofa_results, metadata, output_path):
    """
    Compare latent representations from PLS and MOFA2.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # PLS components 1-2
    axes[0, 0].scatter(
        pls_results['X_scores'][:, 0],
        pls_results['X_scores'][:, 1],
        c=metadata['dose_um'].map({0.1: 'C0', 1.0: 'C1', 10.0: 'C2'}),
        alpha=0.6, s=40, edgecolors='k', linewidth=0.3
    )
    axes[0, 0].set_xlabel('PLS Component 1', fontsize=11)
    axes[0, 0].set_ylabel('PLS Component 2', fontsize=11)
    axes[0, 0].set_title('PLS: Components 1-2', fontweight='bold')
    axes[0, 0].grid(alpha=0.3)
    
    # PLS components 3-4
    axes[0, 1].scatter(
        pls_results['X_scores'][:, 2],
        pls_results['X_scores'][:, 3],
        c=metadata['dose_um'].map({0.1: 'C0', 1.0: 'C1', 10.0: 'C2'}),
        alpha=0.6, s=40, edgecolors='k', linewidth=0.3
    )
    axes[0, 1].set_xlabel('PLS Component 3', fontsize=11)
    axes[0, 1].set_ylabel('PLS Component 4', fontsize=11)
    axes[0, 1].set_title('PLS: Components 3-4', fontweight='bold')
    axes[0, 1].grid(alpha=0.3)
    
    # MOFA factors 1-2
    factors = mofa_results['factors']
    axes[1, 0].scatter(
        factors[:, 0],
        factors[:, 1],
        c=metadata['dose_um'].map({0.1: 'C0', 1.0: 'C1', 10.0: 'C2'}),
        alpha=0.6, s=40, edgecolors='k', linewidth=0.3
    )
    axes[1, 0].set_xlabel('MOFA Factor 1', fontsize=11)
    axes[1, 0].set_ylabel('MOFA Factor 2', fontsize=11)
    axes[1, 0].set_title('MOFA2: Factors 1-2', fontweight='bold')
    axes[1, 0].grid(alpha=0.3)
    
    # MOFA factors 3-4
    axes[1, 1].scatter(
        factors[:, 2],
        factors[:, 3],
        c=metadata['dose_um'].map({0.1: 'C0', 1.0: 'C1', 10.0: 'C2'}),
        alpha=0.6, s=40, edgecolors='k', linewidth=0.3
    )
    axes[1, 1].set_xlabel('MOFA Factor 3', fontsize=11)
    axes[1, 1].set_ylabel('MOFA Factor 4', fontsize=11)
    axes[1, 1].set_title('MOFA2: Factors 3-4', fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    
    plt.suptitle('PLS vs MOFA2: Latent Space Comparison', 
                 y=0.995, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {output_path}")
    plt.close()


plot_method_comparison(
    pls_results,
    mofa_results,
    metadata,
    OUTPUT_DIR / 'figures' / 'pls_mofa_comparison.png'
)

---
## 9. Save Outputs

Save all cleaned matrices and final results.

In [ ]:
# Save aligned feature matrices
print("Saving aligned feature matrices...")

# Imaging matrix (classical + embeddings)
imaging_combined_df = pd.concat([
    classical_features[['sample_key'] + feature_cols_classical],
    embeddings[feature_cols_embeddings]
], axis=1)
imaging_combined_df.to_parquet(
    OUTPUT_DIR / 'data' / 'aligned_imaging_matrix.parquet',
    index=False
)
print(f"  ✓ Saved: {OUTPUT_DIR / 'data' / 'aligned_imaging_matrix.parquet'}")

# Transcriptomics matrix
transcriptomics.to_parquet(
    OUTPUT_DIR / 'data' / 'aligned_transcriptomics_matrix.parquet',
    index=False
)
print(f"  ✓ Saved: {OUTPUT_DIR / 'data' / 'aligned_transcriptomics_matrix.parquet'}")

# Save metadata
metadata.to_csv(OUTPUT_DIR / 'data' / 'sample_metadata.csv', index=False)
print(f"  ✓ Saved: {OUTPUT_DIR / 'data' / 'sample_metadata.csv'}")

print("\n" + "="*60)
print("✓ ANALYSIS COMPLETE")
print("="*60)
print(f"\nAll outputs saved to: {OUTPUT_DIR.absolute()}")
print("\nGenerated files:")
print("\nData:")
for file in sorted((OUTPUT_DIR / 'data').glob('*')):
    print(f"  - {file.name}")
print("\nFigures:")
for file in sorted((OUTPUT_DIR / 'figures').glob('*')):
    print(f"  - {file.name}")

---
## Interpretation Notes

### Workflow Summary

This notebook demonstrates a complete multimodal integration pipeline for high-content imaging and transcriptomics:

1. **Data Sources**: JUMP-CP Cell Painting features (classical + deep embeddings) paired with L1000 transcriptomic profiles

2. **Alignment Strategy**: Used experiment/perturbation ID, plate, well, and replicate metadata to merge modalities

3. **Analysis Methods**:
   - **Correlation Analysis**: Made deep embeddings interpretable by linking to classical features
   - **PLS Regression**: Identified latent components maximizing covariance between modalities
   - **MOFA2**: Discovered shared and modality-specific sources of variation

4. **Key Outputs**:
   - Embedding interpretability scores
   - PLS latent components and loadings
   - MOFA2 factors with variance decomposition
   - Cross-method comparison visualizations

### Adaptation to In-House Data

This workflow can be adapted to any well-based HCI + RNA-seq experiment:

- Replace data loading functions with your own file readers
- Modify `build_sample_key()` to match your metadata structure
- Adjust preprocessing parameters (variance thresholds, PCA components)
- Add domain-specific biological interpretation (pathway enrichment, gene set analysis)

### Technical Considerations

- **Scalability**: For large datasets, subsample features or use mini-batch methods
- **Missing Data**: Current workflow drops incomplete samples; consider imputation for sparse datasets
- **Batch Effects**: Add batch correction (Combat, Harmony) if integrating multi-plate experiments
- **Statistical Testing**: Add permutation tests to assess significance of factor-metadata associations

### Next Steps

1. Validate discovered factors with held-out test perturbations
2. Perform gene set enrichment on top-weighted genes per factor
3. Train predictive models using latent representations
4. Integrate additional modalities (proteomics, metabolomics)
